In [ ]:
from vncorenlp import VnCoreNLP
import re

vncorenlp = VnCoreNLP(
    "VnCoreNLP-1.2.jar",
    annotators="wseg",
    max_heap_size='-Xmx2g'
)
def tokenize_text(text):
    text = text.lower()
    text = re.sub(r"[^\w\s]", " ", text)
    sentences = vncorenlp.tokenize(text)

    tokens = []
    for sent in sentences:
        tokens.extend(sent)

    return tokens
import os

def load_grade_vocab(grade, vocab_dir="grade_vocab"):
    vocab = set()
    for g in range(1, grade + 1):
        path = os.path.join(vocab_dir, f"grade_{g}.txt")
        with open(path, encoding="utf-8") as f:
            for line in f:
                vocab.add(line.strip().lower())
    return vocab
def vocab_coverage(text, vocab):
    tokens = tokenize_text(text)
    if len(tokens) == 0:
        return 0.0

    valid = sum(1 for t in tokens if t in vocab)
    return valid / len(tokens)
import torch
from transformers import MT5ForConditionalGeneration, MT5Tokenizer

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

tokenizer = MT5Tokenizer.from_pretrained("google/mt5-small")
model = MT5ForConditionalGeneration.from_pretrained("google/mt5-small")
model.to(DEVICE)
model.eval()
def abstractive_summary(text, grade, max_output_len=120):
    """
    Sinh tóm tắt diễn giải bằng mT5, có điều kiện cấp lớp
    """

    prompt = (
        f"Tóm tắt văn bản sau cho học sinh lớp {grade}. "
        f"Dùng câu ngắn, từ đơn giản, dễ hiểu:\n{text}"
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512
    ).to(DEVICE)

    summary_ids = model.generate(
        **inputs,
        max_length=max_output_len,
        min_length=30,
        num_beams=4,
        length_penalty=1.0,
        early_stopping=True
    )

    summary = tokenizer.decode(
        summary_ids[0],
        skip_special_tokens=True
    )

    return summary.strip()
def abstractive_summary_with_vocab_control(
    text,
    grade,
    threshold=0.95,
    max_retry=3
):
    vocab = load_grade_vocab(grade)

    best_summary = ""
    best_score = 0.0

    for attempt in range(max_retry):
        summary = abstractive_summary(text, grade)
        score = vocab_coverage(summary, vocab)

        print(f"[Grade {grade}] Attempt {attempt+1} | Coverage: {score:.3f}")

        if score > best_score:
            best_summary = summary
            best_score = score

        if score >= threshold:
            return summary

    return best_summary


In [ ]:
import pandas as pd

data = pd.read_excel(r"E:\Project_NguyenMinhVu_2211110063\Source\datasets\Data DG.xlsx")
df = data[:10]
abstractive_summaries = []

for idx, row in df.iterrows():
    content = str(row["content"])
    grade = int(row["grade"])

    summary = abstractive_summary_with_vocab_control(
        text=content,
        grade=grade,
        threshold=0.95,
        max_retry=3
    )

    abstractive_summaries.append(summary)

df["abstractive_summary"] = abstractive_summaries
df.to_excel(r"E:\Project_NguyenMinhVu_2211110063\Source\datasets\Data_DG_MT5.xlsx", index=False)
